# Environment Setup and Authentication

In [1]:
import sys
import os

# Upgrade Hugging Face ecosystem safely
!{sys.executable} -m pip install --upgrade --quiet transformers accelerate huggingface_hub datasets

import torch
import math
import pandas as pd
import transformers

# Hardware check
device_count = torch.cuda.device_count()
print(f"PyTorch {torch.__version__} | transformers {transformers.__version__} | Detected {device_count} GPUs.")
for i in range(device_count):
    print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")

# Fail withoust falling back to CPU
assert device_count > 0, "No GPU detected. Session options -> Accelerator -> GPU T4 x2, then restart."

from transformers import AutoModelForCausalLM, AutoTokenizer

PyTorch 2.10.0+cu128 | transformers 5.14.1 | Detected 2 GPUs.
  cuda:0 -> Tesla T4
  cuda:1 -> Tesla T4


# Evaluation Metrices

In [2]:
class IntrinsicEvaluator:
    def __init__(self, model, tokenizer, device="cuda:0", max_length=None):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        # Pythia-410M has max_position_embeddings = 2048 if this is not provided cuda fails
        self.max_length = max_length or getattr(model.config, "max_position_embeddings", 2048)

    def get_lengths(self, text: str):
        """Calculates text-intrinsic properties for normalization."""
        if not isinstance(text, str):
            text = str(text)
        num_chars = max(1, len(text))
        num_bytes = max(1, len(text.encode('utf-8')))
        num_words = max(1, len(text.strip().split()))
        return num_chars, num_bytes, num_words

    def evaluate_sequence(self, text: str):
        """Calculates raw NLL and normalizes it into BPB, BPC, and BPW."""
        if not isinstance(text, str):
            text = str(text)

        inputs = self.tokenizer(
            text, return_tensors="pt",
            truncation=True, max_length=self.max_length,
            add_special_tokens=False          # harmonises OPT with the no-BOS models
        ).to(self.device)
        seq_len = inputs["input_ids"].shape[1]

        # seq_len == 1 gives zero scored label positions -> loss is NaN
        # NaN * 0 is still NaN, which would poison the column mean silently
        if seq_len < 2:
            return {"nll": float("nan"), "bpb": float("nan"),
                    "bpc": float("nan"), "bpw": float("nan")}

        # If truncation fired, normalise against the text the model actually
        # scored, otherwise BPB is deflated by bytes never seen by the model.
        if seq_len >= self.max_length:
            text = self.tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)
        chars, bytes_count, words = self.get_lengths(text)

        # inference_mode is strictly cheaper than no_grad
        with torch.inference_mode():
            outputs = self.model(**inputs, labels=inputs["input_ids"])
            # HF CausalLM loss is the average negative log-likelihood per token
            neg_log_likelihood = outputs.loss.item() * (seq_len - 1)

        # Calculate Tokenizer-Independent Metrics
        bpb = neg_log_likelihood / (bytes_count * math.log(2))
        bpc = neg_log_likelihood / (chars * math.log(2))
        bpw = neg_log_likelihood / (words * math.log(2))

        return {
            "nll": neg_log_likelihood,
            "bpb": bpb,
            "bpc": bpc,
            "bpw": bpw
        }
print("IntrinsicEvaluator class defined.")

IntrinsicEvaluator class defined.


# Data Loading and Preprocessing

In [3]:
diverse_sentences_path = "/kaggle/input/datasets/uom220538d/intrinsic-evaluation-dataset/diverse_sentences.csv"
sinhala_mixed_path = "/kaggle/input/datasets/uom220538d/intrinsic-evaluation-dataset/sinhala_mixed_dataset.csv"

try:
    diverse_df = pd.read_csv(diverse_sentences_path).dropna()
    mixed_df = pd.read_csv(sinhala_mixed_path).dropna()
    print(f"Loaded Diverse Sentences: {len(diverse_df)} parallel pairs.")
    print(f"Loaded Mixed Script: {len(mixed_df)} sentences.")
except Exception as e:
    print(f"Data loading failed. Check your file paths. Error: {e}")

Loaded Diverse Sentences: 500 parallel pairs.
Loaded Mixed Script: 500 sentences.


# Model & Tokenizer Initialization

In [4]:
model_id = "MBZUAI/LaMini-GPT-1.5B"
print(f"Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model natively in fp16 across both GPUs
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map={"": 0},
    dtype=torch.float16,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True
)
model.eval()
model.config.use_cache = False   # no generation here; skips KV allocation

# Initialize our custom evaluator
evaluator = IntrinsicEvaluator(model, tokenizer)

print(f"Model loaded.")
print(f"Precision: {model.dtype} | Context: {evaluator.max_length}")

Loading MBZUAI/LaMini-GPT-1.5B...


config.json:   0%|          | 0.00/984 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/788 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/581 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

[transformers] GPT2LMHeadModel LOAD REPORT from: MBZUAI/LaMini-GPT-1.5B
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...47}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded.
Precision: torch.float16 | Context: 1024


# Pilot Run

In [5]:
import csv

pilot_output_path = 'pilot_intrinsic_results.csv'
print("Starting Pilot Run (First 20 items)...")

# Open CSV in append mode to protect against runtime crashes
with open(pilot_output_path, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    # Write standardized header
    writer.writerow([
        'item_index', 
        'unicode_nll', 'unicode_bpb', 'unicode_bpc', 'unicode_bpw',
        'romanized_nll', 'romanized_bpb', 'romanized_bpc', 'romanized_bpw'
    ])
    
    # Slice the first 20 rows of our loaded Kaggle dataframe
    pilot_sample = diverse_df.head(20)
    
    for counter, (index, row) in enumerate(pilot_sample.iterrows(), start=1):
        # 1. Evaluate Native Unicode
        uni_metrics = evaluator.evaluate_sequence(row['sinhala_unicode'])
        
        # 2. Evaluate Romanized (Singlish)
        rom_metrics = evaluator.evaluate_sequence(row['sinhala_romanized'])
        
        # Log to disk
        writer.writerow([
            index, 
            uni_metrics['nll'], uni_metrics['bpb'], uni_metrics['bpc'], uni_metrics['bpw'],
            rom_metrics['nll'], rom_metrics['bpb'], rom_metrics['bpc'], rom_metrics['bpw']
        ])
        
        if counter % 5 == 0:
            print(f"Processed {counter}/20 parallel pairs...")

print(f"\n Pilot run complete. Results saved to {pilot_output_path}")

# Display verification preview
preview_df = pd.read_csv(pilot_output_path)
print("\nFirst 3 rows of calculated metrics:")
display(preview_df.head(3))

# fp16 sanity gate: stop here if the forward pass produced NaN/inf.
import numpy as np
vals = preview_df.drop(columns=['item_index']).to_numpy(dtype=float)
assert np.isfinite(vals).all(), "Non-finite values in pilot output -- inspect before the full run."
print(f"\nPilot means -> Unicode BPB {preview_df['unicode_bpb'].mean():.4f} | "
      f"Romanized BPB {preview_df['romanized_bpb'].mean():.4f}")

Starting Pilot Run (First 20 items)...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Processed 5/20 parallel pairs...
Processed 10/20 parallel pairs...
Processed 15/20 parallel pairs...
Processed 20/20 parallel pairs...

 Pilot run complete. Results saved to pilot_intrinsic_results.csv

First 3 rows of calculated metrics:


,item_index,unicode_nll,unicode_bpb,unicode_bpc,unicode_bpw,romanized_nll,romanized_bpb,romanized_bpc,romanized_bpw
0,0,127.260145,2.353815,6.119919,30.599597,99.609683,3.781747,3.781747,23.951066
1,1,100.565774,2.198269,6.045239,36.271436,71.022508,3.659422,3.659422,25.615955
2,2,121.794879,2.117023,5.668157,29.285478,95.574778,3.830146,3.830146,22.980876



Pilot means -> Unicode BPB 2.5026 | Romanized BPB 3.9095


# Full Intrinsic Evaluation

In [6]:
import csv
import pandas as pd
from tqdm import tqdm

# Evaluate Parallel Diverse Sentences (Unicode vs Romanized)
full_diverse_output = 'lamini_gpt_1_5b_diverse_intrinsic.csv'
print(f"Starting full parallel evaluation on {len(diverse_df)} items...")

with open(full_diverse_output, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow([
        'item_index',
        'unicode_nll', 'unicode_bpb', 'unicode_bpc', 'unicode_bpw',
        'romanized_nll', 'romanized_bpb', 'romanized_bpc', 'romanized_bpw'
    ])

    for index, row in tqdm(diverse_df.iterrows(), total=len(diverse_df), desc="Processing Parallel Pairs"):
        uni_metrics = evaluator.evaluate_sequence(row['sinhala_unicode'])
        rom_metrics = evaluator.evaluate_sequence(row['sinhala_romanized'])

        writer.writerow([
            index,
            uni_metrics['nll'], uni_metrics['bpb'], uni_metrics['bpc'], uni_metrics['bpw'],
            rom_metrics['nll'], rom_metrics['bpb'], rom_metrics['bpc'], rom_metrics['bpw']
        ])

print(f"Diverse evaluation complete. Saved to {full_diverse_output}")

# Evaluate Authentic Mixed-Script Sentences
full_mixed_output = 'lamini_gpt_1_5b_mixed_intrinsic.csv'
mixed_text_col = 'text' if 'text' in mixed_df.columns else mixed_df.columns[0]
print(f"\nStarting full mixed-script evaluation on {len(mixed_df)} items (column: '{mixed_text_col}')...")

with open(full_mixed_output, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(['item_index', 'mixed_nll', 'mixed_bpb', 'mixed_bpc', 'mixed_bpw'])

    for index, row in tqdm(mixed_df.iterrows(), total=len(mixed_df), desc="Processing Mixed Script"):
        mix_metrics = evaluator.evaluate_sequence(row[mixed_text_col])
        writer.writerow([
            index,
            mix_metrics['nll'], mix_metrics['bpb'], mix_metrics['bpc'], mix_metrics['bpw']
        ])

print(f"Mixed-script evaluation complete. Saved to {full_mixed_output}")

# Aggregate and Display Unicode vs Romanized Summary
def calculate_full_aggregates(csv_path, model_label="MBZUAI/LaMini-GPT-1.5B"):
    df = pd.read_csv(csv_path)

    summary = {
        'Metric': ['Bits-Per-Byte (BPB)', 'Bits-Per-Character (BPC)', 'Bits-Per-Word (BPW)'],
        'Unicode (Native)': [
            df['unicode_bpb'].mean(),
            df['unicode_bpc'].mean(),
            df['unicode_bpw'].mean()
        ],
        'Romanized (Singlish)': [
            df['romanized_bpb'].mean(),
            df['romanized_bpc'].mean(),
            df['romanized_bpw'].mean()
        ]
    }

    results_df = pd.DataFrame(summary)
    results_df['Degradation Factor'] = results_df['Romanized (Singlish)'] / results_df['Unicode (Native)']

    print("==========================================================")
    print(f"      {model_label} INTRINSIC EVALUATION SUMMARY")
    print("==========================================================")
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("==========================================================")

    styled = results_df.style.format({
        'Unicode (Native)': '{:.4f}',
        'Romanized (Singlish)': '{:.4f}',
        'Degradation Factor': '{:.2f}×'
    }).set_caption(f"{model_label} — Unicode vs Romanized Sinhala").background_gradient(
        subset=['Degradation Factor'], cmap='Reds'
    )
    display(styled)

    return results_df

full_summary = calculate_full_aggregates(full_diverse_output)

Starting full parallel evaluation on 500 items...




Processing Parallel Pairs:   0%|          | 0/500 [00:00<?, ?it/s]

Processing Parallel Pairs:   0%|          | 2/500 [00:00<00:35, 13.86it/s]

Processing Parallel Pairs:   1%|          | 4/500 [00:00<00:37, 13.28it/s]

Processing Parallel Pairs:   1%|          | 6/500 [00:00<00:37, 13.03it/s]

Processing Parallel Pairs:   2%|▏         | 8/500 [00:00<00:36, 13.53it/s]

Processing Parallel Pairs:   2%|▏         | 10/500 [00:00<00:36, 13.38it/s]

Processing Parallel Pairs:   2%|▏         | 12/500 [00:00<00:36, 13.52it/s]

Processing Parallel Pairs:   3%|▎         | 14/500 [00:01<00:36, 13.44it/s]

Processing Parallel Pairs:   3%|▎         | 16/500 [00:01<00:44, 10.94it/s]

Processing Parallel Pairs:   4%|▎         | 18/500 [00:01<01:01,  7.88it/s]

Processing Parallel Pairs:   4%|▍         | 19/500 [00:01<01:04,  7.41it/s]

Processing Parallel Pairs:   4%|▍         | 20/500 [00:02<01:09,  6.93it/s]

Processing Parallel Pairs:   4%|▍         | 21/500 [00:02<01:31,  5.23it/s]

Processing

Diverse evaluation complete. Saved to lamini_gpt_1_5b_diverse_intrinsic.csv

Starting full mixed-script evaluation on 500 items (column: 'text')...


Processing Mixed Script: 100%|██████████| 500/500 [00:16<00:00, 30.28it/s]

Mixed-script evaluation complete. Saved to lamini_gpt_1_5b_mixed_intrinsic.csv
      MBZUAI/LaMini-GPT-1.5B INTRINSIC EVALUATION SUMMARY
                  Metric  Unicode (Native)  Romanized (Singlish)  Degradation Factor
     Bits-Per-Byte (BPB)            2.4293                3.8568              1.5876
Bits-Per-Character (BPC)            6.4258                3.8660              0.6016
     Bits-Per-Word (BPW)           34.4715               23.8626              0.6922


,Metric,Unicode (Native),Romanized (Singlish),Degradation Factor
0,Bits-Per-Byte (BPB),2.4293,3.8568,1.59×
1,Bits-Per-Character (BPC),6.4258,3.8660,0.60×
2,Bits-Per-Word (BPW),34.4715,23.8626,0.69×


In [7]:
def calculate_mixed_aggregates(csv_path, model_label="MBZUAI/LaMini-GPT-1.5B"):
    df = pd.read_csv(csv_path)

    summary = {
        'Metric': ['Bits-Per-Byte (BPB)', 'Bits-Per-Character (BPC)', 'Bits-Per-Word (BPW)'],
        'Mixed-Script': [
            df['mixed_bpb'].mean(),
            df['mixed_bpc'].mean(),
            df['mixed_bpw'].mean()
        ],
        'Std Dev': [
            df['mixed_bpb'].std(),
            df['mixed_bpc'].std(),
            df['mixed_bpw'].std()
        ]
    }

    results_df = pd.DataFrame(summary)

    print("==========================================================")
    print(f"      {model_label} MIXED-SCRIPT INTRINSIC SUMMARY")
    print(f"      ({len(df)} sentences)")
    print("==========================================================")
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("==========================================================")

    styled = results_df.style.format({
        'Mixed-Script': '{:.4f}',
        'Std Dev': '{:.4f}'
    }).set_caption(f"{model_label} — Mixed-Script Sinhala")
    display(styled)

    return results_df

mixed_summary = calculate_mixed_aggregates(full_mixed_output)

      MBZUAI/LaMini-GPT-1.5B MIXED-SCRIPT INTRINSIC SUMMARY
      (500 sentences)
                  Metric  Mixed-Script  Std Dev
     Bits-Per-Byte (BPB)        2.8883   0.4174
Bits-Per-Character (BPC)        5.6116   0.6571
     Bits-Per-Word (BPW)       34.1073   5.4594


,Metric,Mixed-Script,Std Dev
0,Bits-Per-Byte (BPB),2.8883,0.4174
1,Bits-Per-Character (BPC),5.6116,0.6571
2,Bits-Per-Word (BPW),34.1073,5.4594
